In [0]:
spark.sql("SHOW TABLES IN dbw_retails.bronze;").show()

+--------+--------------------+-----------+
|database|           tableName|isTemporary|
+--------+--------------------+-----------+
|  bronze|eventhub_transact...|      false|
|  bronze|   file_transactions|      false|
+--------+--------------------+-----------+



In [0]:
spark.sql("SELECT COUNT(*) FROM dbw_retails.bronze.eventhub_transactions;").show()

+--------+
|count(1)|
+--------+
|   45000|
+--------+



In [0]:
spark.sql("SELECT * FROM dbw_retails.bronze.eventhub_transactions LIMIT 20;").display()

InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,_ingest_ts,invoice_ts
573415,22809,SET OF 6 T-LIGHTS SANTA,1,10/30/2011 16:17,2.95,13607.0,United Kingdom,2026-03-08T10:32:55.75Z,2011-10-30T16:17:00Z
544837,22087,PAPER BUNTING WHITE LACE,1,2/24/2011 10:29,2.95,15910.0,United Kingdom,2026-03-08T10:32:55.75Z,2011-02-24T10:29:00Z
568151,23163,REGENCY SUGAR TONGS,8,9/25/2011 11:39,2.49,17506.0,United Kingdom,2026-03-08T10:32:55.75Z,2011-09-25T11:39:00Z
568909,22227,HANGING HEART MIRROR DECORATION,24,9/29/2011 13:38,0.65,16818.0,United Kingdom,2026-03-08T10:32:55.75Z,2011-09-29T13:38:00Z
572461,82494L,WOODEN FRAME ANTIQUE WHITE,1,10/24/2011 12:56,2.95,17344.0,United Kingdom,2026-03-08T10:32:55.75Z,2011-10-24T12:56:00Z
578277,21731,RED TOADSTOOL LED NIGHT LIGHT,3,11/23/2011 13:51,1.65,12723.0,France,2026-03-08T10:32:55.75Z,2011-11-23T13:51:00Z
541789,22788,BROCANTE COAT RACK,13,1/21/2011 13:07,8.5,14088.0,United Kingdom,2026-03-08T10:32:55.75Z,2011-01-21T13:07:00Z
559807,23147,SINGLE ANTIQUE ROSE HOOK IVORY,3,7/12/2011 14:48,1.45,17841.0,United Kingdom,2026-03-08T10:32:55.75Z,2011-07-12T14:48:00Z
544339,21977,PACK OF 60 PINK PAISLEY CAKE CASES,120,2/18/2011 9:04,0.42,15228.0,United Kingdom,2026-03-08T10:32:55.75Z,2011-02-18T09:04:00Z
569261,22992,REVOLVER WOODEN RULER,3,10/3/2011 11:34,1.95,17321.0,United Kingdom,2026-03-08T10:32:55.75Z,2011-10-03T11:34:00Z


In [0]:
spark.sql("""SELECT
MAX(_ingest_ts) AS latest_ingest
FROM dbw_retails.bronze.eventhub_transactions;""").display()

latest_ingest
2026-03-08T10:37:51.091Z


In [0]:
spark.sql("""
            SELECT
COUNT(*) AS null_invoice
FROM dbw_retails.bronze.eventhub_transactions
WHERE InvoiceNo IS NULL;
          """).display()

null_invoice
0


In [0]:
spark.sql("""
            SELECT
DATE(_ingest_ts) AS ingest_date,
COUNT(*) AS rows
FROM dbw_retails.bronze.eventhub_transactions
GROUP BY DATE(_ingest_ts)
ORDER BY ingest_date DESC;
          """).display()

ingest_date,rows
2026-03-08,45000


In [0]:
from metrics_utils import log_pipeline_metric

df_test = spark.range(100)

log_pipeline_metric(spark, "bronze_eventhub", df_test)

In [0]:
spark.sql("""
SELECT *
FROM dbw_retails.monitoring.pipeline_metrics
ORDER BY processed_at DESC
""").display()

pipeline,pipeline_stage,row_count,processed_at
bronze_eventhub,null,100,2026-03-08T10:45:15.70959Z
gold_customer,null,4339,2026-03-08T10:41:14.786523Z
gold_country,null,302,2026-03-08T10:41:09.056916Z
gold_product,null,4184,2026-03-08T10:41:02.882942Z
silver_unified,null,0,2026-03-08T10:40:31.112544Z
silver_unified,null,530206,2026-03-08T10:40:16.084047Z
bronze_files,null,541909,2026-03-08T10:38:50.111506Z
bronze_eventhub,null,4332,2026-03-08T10:37:54.943853Z
bronze_eventhub,null,40668,2026-03-08T10:37:49.406844Z


In [0]:
spark.sql("""
SELECT
    MIN(_ingest_ts),
    MAX(_ingest_ts),
    COUNT(*)
FROM dbw_retails.bronze.eventhub_transactions
""").display()

min(_ingest_ts),max(_ingest_ts),count(1)
2026-03-06T13:52:01.289Z,2026-03-07T09:32:45.798Z,30000


In [0]:
spark.sql("""
          SELECT *
FROM dbw_retails.monitoring.pipeline_metrics
WHERE pipeline = 'bronze_eventhub'
ORDER BY processed_at DESC;
          """).display()

pipeline,pipeline_stage,row_count,processed_at
bronze_eventhub,null,100,2026-03-08T10:45:15.70959Z
bronze_eventhub,null,4332,2026-03-08T10:37:54.943853Z
bronze_eventhub,null,40668,2026-03-08T10:37:49.406844Z


In [0]:
bronze_count = spark.sql("""
SELECT COUNT(*) as cnt
FROM dbw_retails.bronze.eventhub_transactions
""").collect()[0]["cnt"]
print(bronze_count)
assert bronze_count > 0, "Bronze ingestion failed!"

45000


In [0]:
spark.sql("""
SELECT COUNT(*)
FROM dbw_retails.bronze.eventhub_transactions
""").show()

+--------+
|count(1)|
+--------+
|   45000|
+--------+

